# Day 10: Linear Regression for Temperature Prediction

## Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load and inspect the dataset

In [ ]:
weather_data = pd.read_csv('../Day-8/cleaned_weather.csv')

print('Dataset shape:', weather_data.shape)
display(weather_data.head())
print('Available columns:')
print(weather_data.columns.tolist())
print('Data types:')
print(weather_data.dtypes)

## Select features and create seasonal features

In [ ]:
baseline_feature_columns = ['humidity', 'wind_speed_km_h', 'wind_bearing_degrees', 'visibility_km', 'pressure_millibars']
seasonal_feature_columns = ['month_sin', 'month_cos']
improved_feature_columns = baseline_feature_columns + seasonal_feature_columns
target_column = 'temperature_c'

weather_data['observation_date'] = pd.to_datetime(weather_data['formatted_date'], errors='coerce', utc=True)
weather_data['month_sin'] = np.sin(2 * np.pi * weather_data['observation_date'].dt.month / 12)
weather_data['month_cos'] = np.cos(2 * np.pi * weather_data['observation_date'].dt.month / 12)

model_data = weather_data[improved_feature_columns + [target_column]].replace([np.inf, -np.inf], np.nan).dropna()

print('Rows available for modeling:', model_data.shape[0])
print('Missing values after preparation:', model_data.isna().sum().sum())

## Correlation heatmap

In [ ]:
correlation_columns = baseline_feature_columns + [target_column]
correlation_matrix = model_data[correlation_columns].corr()

plt.figure(figsize=(9, 7))
plt.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(correlation_columns)), correlation_columns, rotation=45, ha='right')
plt.yticks(range(len(correlation_columns)), correlation_columns)
for row_number in range(len(correlation_columns)):
    for column_number in range(len(correlation_columns)):
        plt.text(column_number, row_number, f'{correlation_matrix.iloc[row_number, column_number]:.2f}', ha='center', va='center')
plt.title('Correlation Heatmap of Weather Features and Temperature')
plt.tight_layout()
plt.show()

## Create one reproducible train-test split

In [ ]:
train_indices, test_indices = train_test_split(model_data.index, test_size=0.2, random_state=42)

X_train_baseline = model_data.loc[train_indices, baseline_feature_columns]
X_test_baseline = model_data.loc[test_indices, baseline_feature_columns]
X_train_improved = model_data.loc[train_indices, improved_feature_columns]
X_test_improved = model_data.loc[test_indices, improved_feature_columns]
y_train = model_data.loc[train_indices, target_column]
y_test = model_data.loc[test_indices, target_column]

print('Training rows:', X_train_baseline.shape[0])
print('Testing rows:', X_test_baseline.shape[0])

## Train the baseline and dummy models

In [ ]:
baseline_model = LinearRegression()
baseline_model.fit(X_train_baseline, y_train)
baseline_predictions = baseline_model.predict(X_test_baseline)

dummy_model = DummyRegressor(strategy='mean')
dummy_model.fit(X_train_baseline, y_train)
dummy_predictions = dummy_model.predict(X_test_baseline)

## Interpret baseline regression coefficients

In [ ]:
feature_scaler = StandardScaler()
X_train_standardized = feature_scaler.fit_transform(X_train_baseline)

standardized_model = LinearRegression()
standardized_model.fit(X_train_standardized, y_train)

coefficient_table = pd.DataFrame({
    'feature': baseline_feature_columns,
    'raw_coefficient': baseline_model.coef_,
    'standardized_coefficient': standardized_model.coef_
})
coefficient_table['absolute_standardized_coefficient'] = coefficient_table['standardized_coefficient'].abs()
coefficient_table = coefficient_table.sort_values('absolute_standardized_coefficient', ascending=False)
display(coefficient_table)

## Train the seasonally enhanced model

In [ ]:
improved_model = LinearRegression()
improved_model.fit(X_train_improved, y_train)
improved_predictions = improved_model.predict(X_test_improved)

## Compare model performance

In [ ]:
model_names = ['Dummy mean model', 'Baseline Linear Regression', 'Seasonal Linear Regression']
model_predictions = [dummy_predictions, baseline_predictions, improved_predictions]
comparison_rows = []

for model_name, predictions in zip(model_names, model_predictions):
    comparison_rows.append({
        'model': model_name,
        'MAE': mean_absolute_error(y_test, predictions),
        'RMSE': np.sqrt(mean_squared_error(y_test, predictions)),
        'R2': r2_score(y_test, predictions)
    })

comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.round(4))

baseline_mae = comparison_table.loc[comparison_table['model'] == 'Baseline Linear Regression', 'MAE'].iloc[0]
improved_mae = comparison_table.loc[comparison_table['model'] == 'Seasonal Linear Regression', 'MAE'].iloc[0]
mae_reduction_percentage = (baseline_mae - improved_mae) / baseline_mae * 100
print(f'MAE reduction from seasonal feature engineering: {mae_reduction_percentage:.2f}%')

## Save prediction results

In [ ]:
prediction_results = pd.DataFrame({
    'actual_temperature_c': y_test.to_numpy(),
    'predicted_temperature_c': baseline_predictions,
    'prediction_error_c': y_test.to_numpy() - baseline_predictions,
    'dummy_predicted_temperature_c': dummy_predictions,
    'dummy_prediction_error_c': y_test.to_numpy() - dummy_predictions,
    'improved_predicted_temperature_c': improved_predictions,
    'improved_prediction_error_c': y_test.to_numpy() - improved_predictions
})
prediction_results.to_csv('prediction_results.csv', index=False)
display(prediction_results.head())

## Baseline actual versus predicted temperature

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, baseline_predictions, alpha=0.2, s=12)
axis_minimum = min(y_test.min(), baseline_predictions.min())
axis_maximum = max(y_test.max(), baseline_predictions.max())
plt.plot([axis_minimum, axis_maximum], [axis_minimum, axis_maximum], color='red', linestyle='--')
plt.xlabel('Actual Temperature (C)')
plt.ylabel('Baseline Predicted Temperature (C)')
plt.title('Baseline: Actual vs Predicted Temperature')
plt.tight_layout()
plt.show()

## Residual analysis for the seasonal model


In [ ]:
improved_residuals = y_test.to_numpy() - improved_predictions

plt.figure(figsize=(8, 6))
plt.scatter(improved_predictions, improved_residuals, alpha=0.2, s=12)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Seasonal Model Predicted Temperature (C)')
plt.ylabel('Residual (Actual - Predicted) (C)')
plt.title('Residuals vs Predicted Temperature')
plt.tight_layout()
plt.show()

residual_groups = pd.DataFrame({'predicted_temperature_c': improved_predictions, 'residual_c': improved_residuals})
residual_groups['prediction_quartile'] = pd.qcut(residual_groups['predicted_temperature_c'], 4)
display(residual_groups.groupby('prediction_quartile', observed=True)['residual_c'].mean().to_frame('mean_residual_c').round(4))